# ส่องดูหน้าตาข้อมูล LOB และกระจายตัวของ Spread

In [6]:
import pandas as pd
import numpy as np
import io
import ccxt
import asyncio
import websockets
import json
from datetime import datetime
import zipfile
import os
import glob
import requests

In [7]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns' , None)
pd.set_option('display.max_row' , None)

In [8]:
os.chdir('/Users/zone/Documents/Project/RL/data/raw')

In [9]:
import pandas as pd
import requests
import os

asset = 'BTCUSDT'
def zip_file(start_date, end_date):
    # 🗓️ กำหนดช่วงเวลาที่ต้องการ (เปลี่ยนได้ตามต้องการ)
    # start_date = '2023-08-10'
    # end_date = '2023-08-24'
    date_range = pd.date_range(start=start_date, end=end_date)

    column_names = [
        'aggregate_trade_id', 'price', 'quantity', 
        'first_trade_id', 'last_trade_id', 'timestamp', 
        'is_buyer_maker', 'is_best_match'
    ]

    def download_binance_data(url, save_path):
        # ข้ามการดาวน์โหลดถ้ามีไฟล์อยู่แล้ว (เอาไว้เผื่อเน็ตหลุด จะได้ไม่ต้องโหลดใหม่ตั้งแต่ต้น)
        if os.path.exists(save_path):
            print(f"⏩ ข้ามการโหลด {save_path} (มีไฟล์อยู่แล้ว)")
            return

        response = requests.get(url)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            print(f"✅ Downloaded: {save_path}")
        else:
            print(f"❌ File not found at: {url}")

    print(f"🚀 เริ่มดาวน์โหลดข้อมูล {asset} ตั้งแต่ {start_date} ถึง {end_date}...")

    # วนลูปดาวน์โหลดทีละวัน
    for single_date in date_range:
        year = single_date.strftime('%Y')
        month = single_date.strftime('%m')
        day = single_date.strftime('%d')
        
        url = f"https://data.binance.vision/data/spot/daily/aggTrades/{asset}/{asset}-aggTrades-{year}-{month}-{day}.zip"
        save_path = f"{asset}-{year}-{month}-{day}.zip"
        
        download_binance_data(url, save_path)

In [10]:
zip_file("2023-08-10", "2023-08-24") # Mean-Reverting (Range-bound)
zip_file("2024-02-15", "2024-02-29") # Trending (High Drift)
zip_file("2022-11-01", "2022-11-15") # High Volatility / High Toxicity (Flash Crash)

🚀 เริ่มดาวน์โหลดข้อมูล BTCUSDT ตั้งแต่ 2023-08-10 ถึง 2023-08-24...
⏩ ข้ามการโหลด BTCUSDT-2023-08-10.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-11.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-12.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-13.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-14.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-15.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-16.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-17.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-18.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-19.zip (มีไฟล์อยู่แล้ว)
⏩ ข้ามการโหลด BTCUSDT-2023-08-20.zip (มีไฟล์อยู่แล้ว)
✅ Downloaded: BTCUSDT-2023-08-21.zip
✅ Downloaded: BTCUSDT-2023-08-22.zip
✅ Downloaded: BTCUSDT-2023-08-23.zip
✅ Downloaded: BTCUSDT-2023-08-24.zip
🚀 เริ่มดาวน์โหลดข้อมูล BTCUSDT ตั้งแต่ 2024-02-15 ถึง 2024-02-29...
✅ Downloaded: BTCUSDT-2024-02-15.zip
✅ Downloaded: BTCUSDT-2024-02-16.zip
✅ Downloaded: BTCUSDT-2024-02-17.zip
✅ Downloade

In [12]:
import os
import zipfile
import pandas as pd

# กำหนดชื่อคอลัมน์เผื่อไว้ในกรณีที่ Cell ก่อนหน้าไม่ได้รัน
column_names = [
    'aggregate_trade_id', 'price', 'quantity', 
    'first_trade_id', 'last_trade_id', 'timestamp', 
    'is_buyer_maker', 'is_best_match'
]

def process_regime_to_csv(start_date, end_date, output_csv, asset='BTCUSDT'):
    """
    ดึงเฉพาะไฟล์ Zip ตามช่วงเวลาที่กำหนด แล้วนำมาต่อกันเป็น CSV แยกตามสภาวะตลาด
    """
    date_range = pd.date_range(start=start_date, end=end_date)
    print(f"\n🔎 กำลังสร้างไฟล์ {output_csv} (ตั้งแต่ {start_date} ถึง {end_date})...")
    
    # เช็คว่าถ้ามี CSV เก่าอยู่ให้ลบทิ้งก่อน จะได้ไม่ต่อไฟล์ซ้ำซ้อน
    if os.path.exists(output_csv):
        os.remove(output_csv)
        print(f"🗑️ ลบไฟล์ {output_csv} ตัวเก่าทิ้งแล้ว")

    first_file = True

    for single_date in date_range:
        year = single_date.strftime('%Y')
        month = single_date.strftime('%m')
        day = single_date.strftime('%d')
        
        # คาดเดาชื่อไฟล์ Zip ที่โหลดมาไว้แล้วจาก Cell ก่อนหน้า
        zip_filename = f"{asset}-{year}-{month}-{day}.zip"
        
        if not os.path.exists(zip_filename):
            print(f"⚠️ ไม่พบไฟล์ {zip_filename} ข้ามไปก่อน...")
            continue

        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            csv_filename = zip_ref.namelist()[0]
            with zip_ref.open(csv_filename) as f:
                # 1. โหลดแค่ของวันเดียวเข้า RAM
                df = pd.read_csv(f, names=column_names)
                
                # 2. แปลงข้อมูล
                df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
                df['side'] = df['is_buyer_maker'].map({True: 'SELL', False: 'BUY'})
                
                # 3. เทลงไฟล์ CSV ทันที
                df.to_csv(
                    output_csv, 
                    mode='w' if first_file else 'a', 
                    header=first_file, 
                    index=False
                )
                
                first_file = False
                print(f"🔨 Processed & Appended: {zip_filename} -> {output_csv}")

    print(f"🎉 สร้างไฟล์สำเร็จ! เซฟไว้ที่: {output_csv}")

# ==========================================
# 🚀 สั่งรันฟังก์ชันเพื่อสร้าง CSV 3 ไฟล์แยกกัน
# ==========================================

# 1. สร้างไฟล์ Sideway (Mean-Reverting)
process_regime_to_csv("2023-08-10", "2023-08-24", "BTCUSDT_sideway.csv")

# 2. สร้างไฟล์ Trend (High Drift)
process_regime_to_csv("2024-02-15", "2024-02-29", "BTCUSDT_trend.csv")

# 3. สร้างไฟล์ Toxic (High Volatility / Flash Crash)
process_regime_to_csv("2022-11-01", "2022-11-15", "BTCUSDT_toxic.csv")


🔎 กำลังสร้างไฟล์ BTCUSDT_sideway.csv (ตั้งแต่ 2023-08-10 ถึง 2023-08-24)...
🗑️ ลบไฟล์ BTCUSDT_sideway.csv ตัวเก่าทิ้งแล้ว
🔨 Processed & Appended: BTCUSDT-2023-08-10.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-11.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-12.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-13.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-14.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-15.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-16.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-17.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-18.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-19.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-20.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-08-21.zip -> BTCUSDT_sideway.csv
🔨 Processed & Appended: BTCUSDT-2023-